# 07 -- QuerySoftMax

Проверяю loss, который сравнивает объявления внутри одного запроса. Пул и признаки те же, что у CatBoost.
Все случайные операции используют seed 143.

In [1]:
from pathlib import Path
import sys
import json
import time
import random

import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostRanker, Pool
from IPython.display import display

rk_root = Path.cwd() if (Path.cwd() / 'notebooks').exists() else Path.cwd().parent
import nbformat

# Подключаю только ячейки с функциями; расчеты других ноутбуков не запускаю.
for rk_file in ['02_feature_engineering.ipynb']:
    for rk_cell in nbformat.read(rk_root / 'notebooks' / rk_file, as_version=4).cells:
        if 'definitions' in rk_cell.metadata.get('tags', []):
            exec(rk_cell.source, globals())
rk_out = rk_root / 'outputs'
rk_seed = 143
random.seed(rk_seed)
np.random.seed(rk_seed)

## Обучающие кандидаты

Группы без найденного позитива исключаю только из обучения. На dev они остаются в знаменателе.
Для классификаторов суммарный вес положительной и неразмеченной части каждого запроса одинаковый.
Каталожные ID и target не попадают в матрицу. Масштабирование обучаю только на rank_train.

In [2]:
rk_train = (
    pd.read_parquet(rk_out / 'cache/extended_rank_train.parquet')
    .sort_values(['query_id', 'pos'])
    .reset_index(drop=True)
)
rk_dev = (
    pd.read_parquet(rk_out / 'cache/extended_dev.parquet')
    .sort_values(['query_id', 'pos'])
    .reset_index(drop=True)
)
rk_queries = pd.read_parquet(rk_out / 'cache/eval_queries.parquet')
rk_dev_queries = rk_queries[rk_queries.split.eq('dev')]
rk_items = pd.read_parquet(rk_out / 'cache/items.parquet', columns=['item_id'])
rk_truth = json.loads((rk_out / 'cache/truth.json').read_text())
rk_dev_truth = {q: rk_truth[q] for q in rk_dev_queries.query_id}
rk_ids = rk_items.item_id.to_numpy()
rk_columns = feature_columns(rk_train)
rk_train['target'] = [
    int(rk_ids[p] in rk_truth[q]) for q, p in zip(rk_train.query_id, rk_train.pos)
]
rk_found = rk_train.groupby('query_id').target.transform('sum')
rk_train = rk_train[rk_found.gt(0)].reset_index(drop=True)
rk_npos = rk_train.groupby('query_id').target.transform('sum')
rk_n = rk_train.groupby('query_id').target.transform('size')
rk_weights = np.where(
    rk_train.target.eq(1), 0.5 / rk_npos, 0.5 / (rk_n - rk_npos).clip(lower=1)
)
rk_weights /= rk_weights.mean()
rk_x = model_matrix(rk_train, rk_columns)
rk_xdev = model_matrix(rk_dev, rk_columns)
rk_share = json.loads((rk_out / 'models/hybrid.json').read_text())['warm_share']
rk_results = []
rk_scores = rk_dev[['query_id', 'pos']].copy()
write_json(
    {'features': rk_columns, 'seed': rk_seed}, rk_out / 'models/extended_features.json'
)

In [3]:
def rk_evaluate(name, scores, seconds):
    predictions = predictions_from_frame(
        rk_dev.assign(score=scores), rk_items, score='score', k=100
    )
    per_query = recall_table(predictions, rk_dev_truth).merge(
        rk_dev_queries[['query_id', 'regime']], on='query_id'
    )
    values = per_query.groupby('regime')['recall@50'].mean()
    rk_scores[name] = scores
    rk_results.append(
        {
            'model': name,
            'cold': values['cold'],
            'warm': values['warm'],
            'objective': (1 - rk_share) * values['cold'] + rk_share * values['warm'],
            'fit_seconds': seconds,
        }
    )
    pd.DataFrame(rk_results).to_csv(rk_out / 'validation/models_rk.csv', index=False)
    per_query.to_parquet(rk_out / f'validation/dev_{name}.parquet', index=False)
    print(rk_results[-1], flush=True)

## Обучение

QuerySoftMax сравнивает объявления внутри одного запроса. Это ближе к отбору кандидатов,
чем независимая классификация. Функция потерь не равна Recall@50, поэтому выбор все равно
делаю по собственной метрике со всеми dev-запросами, включая пропущенные retrieval-позитивы.

In [4]:
rk_start = time.perf_counter()
rk_pool = Pool(rk_x, rk_train.target, group_id=rk_train.query_id.to_numpy())
rk_ranker = CatBoostRanker(
    iterations=300,
    depth=6,
    learning_rate=0.07,
    loss_function='QuerySoftMax',
    l2_leaf_reg=5,
    random_seed=rk_seed,
    thread_count=4,
    allow_writing_files=False,
    verbose=100,
)
rk_ranker.fit(rk_pool)
rk_seconds = time.perf_counter() - rk_start
for rk_trees in [150, 300]:
    rk_evaluate(
        f'ranker_{rk_trees}', rk_ranker.predict(rk_xdev, ntree_end=rk_trees), rk_seconds
    )
rk_ranker.save_model(str(rk_out / 'models/extended_ranker.cbm'))

0:	learn: 5.7666589	total: 1.16s	remaining: 5m 47s


100:	learn: 3.0916805	total: 1m 51s	remaining: 3m 39s


200:	learn: 2.8945117	total: 3m 42s	remaining: 1m 49s


299:	learn: 2.7623694	total: 5m 32s	remaining: 0us


{'model': 'ranker_150', 'cold': np.float64(0.7754722222222221), 'warm': np.float64(0.7932380952380952), 'objective': np.float64(0.7821308018721354), 'fit_seconds': 333.10810075000063}


{'model': 'ranker_300', 'cold': np.float64(0.7868055555555555), 'warm': np.float64(0.8020714285714285), 'objective': np.float64(0.792527144993397), 'fit_seconds': 333.10810075000063}


In [5]:
rk_scores.to_parquet(rk_out / 'cache/scores_rk.parquet', index=False)
pd.DataFrame(rk_results).to_csv(rk_out / 'validation/models_rk.csv', index=False)
display(pd.DataFrame(rk_results))

,model,cold,warm,objective,fit_seconds
0,ranker_150,0.775472,0.793238,0.782131,333.108101
1,ranker_300,0.786806,0.802071,0.792527,333.108101


## Вывод

QuerySoftMax дал 0.79253 при 300 деревьях, ниже 0.80617 у Logloss на том же пуле.
Близость функции потерь к ранжированию сама по себе не гарантирует лучший Recall@50.